<a href="https://colab.research.google.com/github/ehdbddl06001-ui/my-github-test/blob/claude%2Fexp-2026-001-q4o-leakage-free-residual/exp-2026-001-q4o-leakage-free-residual/notebooks/quest47_q4o_leakage_free_residual_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXP-2026-001 / Q4-O — leakage-free morphology baseline + current-beat raw-CNN residual

| | |
|---|---|
| Spec | `experiments/specs/EXP-2026-001-q4o-leakage-free-residual-cnn.md` |
| Module | `mit-bih/q4o_leakage_free_residual.py` (all evaluation / fold / OOF / statistics logic) |
| Tests | `mit-bih/test_q4o_leakage_free_residual.py` |
| Data | `/content/drive/MyDrive/mitbih/svdb_data5.npz` — the exact file Q4-N read |
| Runtime | **GPU** (Runtime -> Change runtime type -> GPU) |

This notebook is a **wrapper only**: Drive mount, config, run, visualise, save. It
deliberately contains no evaluation logic, no fold construction, no OOF stacking, and
no statistics — all of that lives in the module so it can be unit-tested.

## Why this run exists

Q4-N built the residual CNN's offset with a function that wrote **both** the train and
the test positions of one shared array, once per fold, in sequence:

```python
sc[tr] = lr.decision_function((X[tr] - mu) / sd)   # in-sample; the next fold overwrites it
sc[te] = lr.decision_function((X[te] - mu) / sd)
```

After the last of five folds, roughly **80%** of that array holds in-sample
predictions. So `cpu_comb = 0.8445`, `boost_fix = 0.8631`, and `boost_rank = 0.8492`
are **not** baselines and **not** improvements. They are carried here only as
contaminated reference values for the Arm E diagnostic.

## The five arms

| Arm | Name | Input | Offset |
|---|---|---|---|
| A | `morph_baseline` | frozen Q4-N morphology, 17 columns | — |
| B | `raw_current_cnn` | current beat, 2 leads, nothing else | — |
| **C** | `morph_plus_raw_residual` | current beat, 2 leads | cross-fitted morph logit |
| D | `shuffled_waveform_control` | current beat, **permuted within record** | cross-fitted morph logit |
| E | `corrected_q4n_diagnostic` | Q4-N 3-beat + 2 RR channels | cross-fitted `comb` logit |

**Primary**: `C − A`. **Key negative control**: `C − D`. Arm E is diagnostic only and
must not be read as a result or a new baseline.

## Pre-registered gates (all six required for PASS)

1. `mean(C − A) >= +0.015`
2. paired record-bootstrap 95% CI lower bound `> 0`
3. `mean(C − D) > 0` and its CI lower bound `> 0`
4. at least 4 of 5 seeds positive
5. lower-tail p10 of C not worse than A by more than `0.01`
6. every leakage / reproducibility assertion passes

**NO-GO does not mean "try a Transformer."** It means keep the morphology baseline and
go back to failure-record and lower-tail analysis. PASS does not mean Transformer
either — it means port the same minimal residual structure to MIT-BIH DS1→DS2.

## 1. Mount Drive and pull the repository code

`REPO_BRANCH` must be the branch carrying this experiment. The module is imported from
the checkout, never pasted into a cell — pasted code cannot be tested.

> **If you are re-running after a fix:** this cell purges the module from `sys.modules`
> before importing and then calls `Q.self_check()` in-process. A plain `import` is a
> no-op when the module is already loaded, so without this a `git pull` would leave the
> kernel executing the **old** code — the traceback would then show new source lines
> against old line numbers, and a fixed bug would look like it never went away. If the
> self-check fails, restart the runtime (Runtime → Restart session) and run again.

In [21]:
import os, sys, subprocess, time, importlib

REPO_URL     = "https://github.com/ehdbddl06001-ui/my-github-test.git"
REPO_BRANCH  = "claude/exp-2026-001-q4o-leakage-free-residual"
REPO_DIR     = "/content/my-github-test"
NEED_VERSION = 4          # minimum q4o module version this notebook requires

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as exc:
    print("not Colab:", exc)
    DRIVE_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT", "/content")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    f"origin/{REPO_BRANCH}"], check=True)

MOD_DIR = os.path.join(REPO_DIR, "mit-bih")
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

# ---------------------------------------------------------------------------
# Force a genuine re-import.
#
# `import q4o_leakage_free_residual` is a NO-OP once the module sits in
# sys.modules. Pulling new code updates the file on disk but NOT the code this
# kernel executes, and the test cell below runs in a subprocess -- so it reads
# the new file and passes while the kernel still runs the old one. That is how a
# fixed bug appears to persist. Purge the module and invalidate the import
# caches so the pull actually takes effect.
# ---------------------------------------------------------------------------
for _name in [m for m in sys.modules if m.startswith("q4o_leakage_free_residual")]:
    del sys.modules[_name]
importlib.invalidate_caches()

import q4o_leakage_free_residual as Q

# In-process proof that the fresh code is live -- runs the exact path that a
# stale import gets wrong (a cohort containing records below MIN_S/MIN_N).
check = Q.self_check(min_version=NEED_VERSION)

print("module     :", check["module_file"])
print("version    :", check["module_version"], "-", check["module_build"])
print("self-check : OK  (", check["n_scorable"], "of", check["n_record"],
      "records scorable,", check["n_unscored_beats"], "beats correctly unscored )")
print("drive root :", DRIVE_ROOT)
print("repo commit:", Q.git_commit_sha(REPO_DIR))
print("packages   :", Q.package_versions())
print("gpu        :", Q.gpu_info())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
module     : /content/my-github-test/mit-bih/q4o_leakage_free_residual.py
version    : 4 - 2026-08-08 q4o.4 — best_epoch=0 semantics corrected in all prose (epoch 0 is post-update, not pre-training); training_history.json joins the immutability fingerprint when present; q4o.3 added the reporting layer
self-check : OK  ( 6 of 8 records scorable, 240 beats correctly unscored )
drive root : /content/drive/MyDrive
repo commit: f9080541fa2789e37bcfe881c1502405fbf3a179
packages   : {'python': '3.12.13', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'numpy': '2.0.2', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'pandas': '2.2.2', 'torch': '2.11.0+cu128', 'matplotlib': '3.10.0'}
gpu        : {'cuda_available': True, 'device_name': 'Tesla T4', 'cuda_version': '12.8', 'device_count': 1}


## 2. Run the tests first

If any test fails, stop. A failing leakage assertion invalidates the run before it
starts — that is criterion 6.

In [22]:
rc = subprocess.run(
    [sys.executable, os.path.join(REPO_DIR, "mit-bih",
                                  "test_q4o_leakage_free_residual.py")],
    capture_output=True, text=True)
print(rc.stdout[-4000:])
if rc.returncode != 0:
    print(rc.stderr[-3000:])
    raise SystemExit("tests failed - do not run the experiment")

# The line above proves the FILE is good. It says nothing about the module this
# kernel has loaded, because it ran in a separate process. Re-assert in-kernel.
assert Q.MODULE_VERSION >= NEED_VERSION, (
    f"tests passed against the file, but this kernel holds q4o version "
    f"{Q.MODULE_VERSION}. Re-run the cell above, or restart the runtime.")
Q.self_check(min_version=NEED_VERSION)
print(f"tests passed · in-kernel module version {Q.MODULE_VERSION} verified")

figures/achievement_by_k.png
  PASS  report produced figures/seed_effects.png
  PASS  report produced figures/fold_training_diagnostics.png
  PASS  report produced figures/patient_delta_waterfall.png
  PASS  report produced figures/patient_delta.csv
  PASS  report produced figures/metric_distribution.png
  PASS  report produced figures/report_summary.md
  PASS  the old mixed-axis contrasts.png is gone — primary and reference contrasts are on separate axes
  PASS  the run itself writes the split contrasts_primary.png
  PASS  the run itself writes the split contrasts_reference.png
  PASS  report_summary.md covers 'Executive Summary'
  PASS  report_summary.md covers '0.8631'
  PASS  report_summary.md covers 'baseline'
  PASS  report_summary.md covers 'NO-GO'
  PASS  report_summary.md covers 'PASS'

------------------------------------------------------------------------------
15. reporting — never fabricates a missing training history
------------------------------------------------------

## 3. Config

The data path is **fixed to `svdb_data5.npz`** — the exact file Q4-N read. Do not point
this at `svdb_data.npz`; it is a different, older file without `y3`/`sym`, and the
loader will refuse it. If the file is not where this cell expects it, stop and report
a blocker rather than substituting another file.

In [23]:
# ---------------------------------------------------------------------------
# ANALYZE_EXISTING_RUN
#   True  -> no training at all. Reads an existing run bundle from EXISTING_RUN_DIR
#            and renders the full report. Nothing measured is recomputed or altered.
#   False -> run the experiment, then report on it.
# ---------------------------------------------------------------------------
ANALYZE_EXISTING_RUN = True
EXISTING_RUN_DIR     = "20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn"

DATA_PATH  = os.path.join(DRIVE_ROOT, "mitbih", "svdb_data5.npz")
PROJECT    = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
RUNS_DIR   = os.path.join(PROJECT, "runs")
REGISTRY   = os.path.join(PROJECT, "registry.jsonl")

SEEDS      = list(Q.TRAIN_SEEDS)   # five training seeds, pre-registered
EPOCHS     = Q.DL_EPOCH
BATCH      = Q.DL_BATCH
N_BOOT     = Q.NB_BOOT
PORT_CHECK = True                  # re-score Arm A under Q4-N's LORO (fidelity check)

if ANALYZE_EXISTING_RUN:
    OUT_DIR = (EXISTING_RUN_DIR if os.path.isabs(EXISTING_RUN_DIR)
               else os.path.join(RUNS_DIR, EXISTING_RUN_DIR))
    assert os.path.isdir(OUT_DIR), f"run bundle not found: {OUT_DIR}"
    print("MODE     : ANALYZE_EXISTING_RUN - no training, report only")
    print("run dir  :", OUT_DIR)
else:
    TIMESTAMP = time.strftime("%Y%m%dT%H%M", time.gmtime())
    OUT_DIR   = os.path.join(RUNS_DIR, Q.run_dir_name(TIMESTAMP))
    assert os.path.exists(DATA_PATH), (
        f"{DATA_PATH} not found. Do NOT substitute svdb_data.npz — stop and report a "
        f"blocker.")
    print("MODE     : FULL RUN")
    print("data     :", DATA_PATH)
    print("out dir  :", OUT_DIR)
    print("seeds    :", SEEDS)

print("k-sweep  :", Q.K_SWEEP, " operating points:", Q.K_OP)
print("gates    : gain >=", Q.GATE_MIN_GAIN, "| seeds >=", Q.GATE_MIN_SEED_AGREE,
      "| lower-tail drop <=", Q.GATE_LOWER_TAIL_MAX_DROP)

MODE     : ANALYZE_EXISTING_RUN - no training, report only
run dir  : /content/drive/MyDrive/MedKOS/ecg-model/runs/20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn
k-sweep  : (50, 100, 200, 300)  operating points: (30, 50)
gates    : gain >= 0.015 | seeds >= 4 | lower-tail drop <= 0.01


## 4. Load the cohort and record its provenance

Everything the manifest needs — absolute path, SHA256, shapes, dtypes, class and
record counts — is captured here, before any modelling.

In [24]:
if ANALYZE_EXISTING_RUN:
    print("skipped - ANALYZE_EXISTING_RUN reads the finished bundle instead.")
    cohort = provenance = None
else:
    cohort, provenance = Q.load_cohort(DATA_PATH)

    print("file      :", provenance["file_name"])
    print("sha256    :", provenance["sha256"])
    print("beats     :", provenance["n_sample_labelled"], "of",
          provenance["n_sample_total"])
    print("shape     :", provenance["arrays"]["beat"]["shape"],
          provenance["arrays"]["beat"]["dtype"])
    print("classes   :", provenance["class_counts"])
    print("records   :", provenance["n_record"], "(record == patient:",
          provenance["record_equals_patient"], ")")

    rec_ok = Q.scorable_records(cohort)
    burden = Q.record_burden(cohort, rec_ok)
    fold_map = Q.make_fold_map(rec_ok, burden)
    Q.assert_fold_map_partition(fold_map, rec_ok, Q.N_OUTER_FOLDS)
    print("scorable  :", len(rec_ok), "records (MIN_S =", Q.MIN_S,
          ", MIN_N =", Q.MIN_N, ")")
    for f in range(Q.N_OUTER_FOLDS):
        rs = sorted(r for r in rec_ok if fold_map[r] == f)
        print(f"  fold {f}: {len(rs):2d} records  {rs}")

skipped - ANALYZE_EXISTING_RUN reads the finished bundle instead.


## 5. Run

Every arm, every seed, every fold. The runner raises on any leakage violation rather
than reporting a number, so reaching the end is itself part of criterion 6.

This is 5 seeds × 4 neural arms × 5 folds = **100 model trainings**. Budget
**1–3 hours** on a Colab GPU and keep the tab alive; the per-batch bottleneck is
CPU-side fancy indexing of the waveform array, not the GPU.

Peak host memory is roughly 3–4 GB (Arm E's 3-beat input is ~1.8 GB on its own), which
fits a standard Colab runtime but leaves little headroom — restart the runtime before
running if you have already loaded other large arrays.

Note that 22 of SVDB's 78 records fall below `MIN_S`/`MIN_N` and are not scored. Their
beats stay in the cohort, are absent from the fold map, and appear as `NaN` in
`probs.npy` with `fold = -1` and `scored_mask = False` in `predictions.npz`. That is
by design, not a failure.

In [25]:
if ANALYZE_EXISTING_RUN:
    print("skipped - no training in ANALYZE_EXISTING_RUN mode.")
    import json
    result = json.load(open(os.path.join(OUT_DIR, "result.json"), encoding="utf-8"))
    print("loaded the existing result.json · verdict:", result["gates"]["verdict"])
else:
    log = Q.RunLog()
    result = Q.run_experiment(
        cohort, provenance, OUT_DIR,
        seeds=SEEDS, epochs=EPOCHS, batch=BATCH, n_boot=N_BOOT,
        port_check=PORT_CHECK, smoke=False, log=log)
    print("\nverdict:", result["gates"]["verdict"])

skipped - no training in ANALYZE_EXISTING_RUN mode.
loaded the existing result.json · verdict: NO-GO


In [26]:
if ANALYZE_EXISTING_RUN:
    print("skipped - the registry already carries this run; reporting never re-appends.")
else:
    record = {
        "run_id": Q.run_dir_name(TIMESTAMP),
        "experiment_id": Q.EXPERIMENT_ID,
        "arm_id": Q.ARM_ID,
        "primary_metric": result["primary_metric"],
        "primary_value": result["contrasts"]["C_minus_A"]["record_bootstrap"]["mean"],
        "primary_ci": [result["contrasts"]["C_minus_A"]["record_bootstrap"]["ci_low"],
                       result["contrasts"]["C_minus_A"]["record_bootstrap"]["ci_high"]],
        "negative_control": result["contrasts"]["C_minus_D"]["record_bootstrap"]["mean"],
        "verdict": result["gates"]["verdict"],
        "conclusion": (
            f"C-A {result['contrasts']['C_minus_A']['record_bootstrap']['mean']:+.4f}, "
            f"C-D {result['contrasts']['C_minus_D']['record_bootstrap']['mean']:+.4f}, "
            f"{result['gates']['verdict']}"),
        "run_folder": OUT_DIR,
        "data_sha256": provenance["sha256"],
        "git_commit": Q.git_commit_sha(REPO_DIR),
    }
    Q.append_registry(REGISTRY, record)
    print(json.dumps(record, indent=2))

skipped - the registry already carries this run; reporting never re-appends.


## 6. 보고서 생성 (presentation only)

`Q.generate_report()` 는 완성된 run 번들을 **읽기만** 한다. 학습하지 않고, arm·fold·
seed·지표·bootstrap·gate 를 건드리지 않으며, `result.json` / `manifest.json` /
`config.json` / `fold_map.json` / `predictions.npz` / `arms/*/probs.npy` 에 쓰지 않는다.

두 가지를 스스로 검증한다.

1. **재현** — 저장된 logit 에서 per-record k-sweep 을 다시 계산해 `result.json` 의
   요약값과 일치하는지 확인한다. 어긋나면 보고서를 만들지 않고 예외를 던진다.
2. **불변** — 위 파일들의 SHA256 을 보고서 생성 전후로 비교한다. 하나라도 바뀌면 예외.

In [27]:
report = Q.generate_report(OUT_DIR)

print()
print("재현 확인 :", report["reconciliation"])
print("측정 산출물 불변 :", report["fingerprint_stable"])
print("training history :",
      "있음" if report["training_history_present"] else "없음 (이 run 은 기록 기능 이전)")

[    0.4s] reporting on 20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn — verdict NO-GO, 5 seeds, 56 records
[    0.8s] reconciled against result.json — max abs diff 0.000e+00
[    5.7s] no training_history.json in this run — learning curves skipped, nothing fabricated
[    5.7s] wrote 11 report artifacts to /content/drive/MyDrive/MedKOS/ecg-model/runs/20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn/figures
[    5.8s] verified — no measured artifact was modified by reporting

재현 확인 : {'max_abs_diff': 0.0, 'within_tolerance': True, 'tolerance': 1e-09, 'n_checked': 30}
측정 산출물 불변 : True
training history : 없음 (이 run 은 기록 기능 이전)


## 7. Executive Summary

사람이 가장 먼저 읽어야 할 부분. 모든 수치는 `result.json` 에서 그대로 가져온다.

In [28]:
print(report["executive_summary_ko"])

  EXP-2026-001 / Q4-O — Executive Summary
  run: 20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn

■ 최종 판정: NO-GO
   사전 등록된 6개 gate 중 3개 통과 / 3개 실패.  (판정 기준은 실행 전에 고정되었고, 결과를 보고 바꾸지 않았다.)

■ morphology baseline (Arm A) — 이번 실험이 지켜낸 기준선
   k-sweep 달성률 평균  0.8310   (record 매크로 PR-AUC 0.7166 · AUROC 0.9483)
   하위꼬리 p10 0.5577 · 최악 레코드 61 (0.3798)
   ※ 이 값은 누수 없는 5-fold record-grouped CV 에서 측정됐다. Q4-N 의 0.8445/0.8631/0.8492 와 같은 자리에 두고 비교하면 안 된다.

■ 주 비교 — C(형태+원파형 잔차) − A(형태 단독)
   +0.0009 [95% CI -0.0021, +0.0042]
   seed 4/5 개에서 양(+) 방향 · 통과 기준은 평균 ≥ +0.015 이고 CI 하한 > 0

■ 음성 대조 — C − D(파형을 레코드 안에서 셔플한 동일 구조)
   +0.0013 [95% CI -0.0016, +0.0046]
   이 대비가 0 이면, C 가 얻은 것은 '박동 단위 파형 정보'가 아니다.

■ 진단 — E − cleanComb (Q4-N boost_fix 구조에서 잔차만의 효과)
   +0.0037 [95% CI -0.0035, +0.0118]

■ 통과한 gate
   ✅ 4_seed_direction_stable
   ✅ 5_lower_tail_not_worse
   ✅ 6_leakage_and_reproducibility
■ 실패한 gate
   ❌ 1_mean_gain_ge_0.015
   ❌ 2_ci_lower_gt_0
   ❌ 3_beats_shuffle_control

■ 이 결과가 의미하는 

## 8. 표와 그림

그림의 축 라벨은 영어다 — Colab 에 기본 CJK 폰트가 없어 한글이 두부(□)로 깨진다.
해석은 각 그림 아래에 한국어로 붙인다.

In [29]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Markdown
import pandas as pd

FIG = os.path.join(OUT_DIR, "figures")

def show(name, caption):
    """Render one report figure with its Korean interpretation underneath."""
    path = os.path.join(FIG, name)
    if not os.path.exists(path):
        display(Markdown(f"*(`{name}` 없음 — 이 run 에서는 생성되지 않았다)*"))
        return
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(14, 14 * h / w))
    ax.imshow(img); ax.axis("off")
    plt.show()
    display(Markdown(caption))

bundle = Q.load_run_bundle(OUT_DIR)
res    = bundle.result
ca     = res["contrasts"]["C_minus_A"]["record_bootstrap"]
cd     = res["contrasts"]["C_minus_D"]["record_bootstrap"]
a_ksw  = res["arms"][Q.ARM_A]["seed_averaged_ksw"]
c_ksw  = res["arms"][Q.ARM_C]["seed_averaged_ksw"]
stall  = Q.training_stalled(bundle, Q.ARM_C)
n_seed = len(bundle.seeds)

In [30]:
display(Markdown("### 1. arm 요약표"))
display(pd.read_csv(os.path.join(FIG, "arm_metrics.csv")))
show("arm_summary_table.png",
     f"**해석.** morphology baseline(A) 의 k-sweep 달성률은 **{a_ksw['mean']:.4f}**, "
     f"주 비교군 C 는 **{c_ksw['mean']:.4f}** 다. "
     f"`Δ vs A` 열은 같은 record·같은 seed 로 짝지은 대비이며, 짝지은 대비가 없는 "
     f"arm 만 평균의 차이(\\*)로 표기했다. "
     f"seed SD 열은 seed 간 변동으로, 이 값보다 작은 차이는 해석하지 않는다.")

### 1. arm 요약표

,arm,short,ksw_mean,delta_vs_A,delta_source,prauc,auroc,p10,worst,worst_record,seed_sd
0,morph_baseline,A,0.830955,0.000000,baseline,0.716615,0.948328,0.557703,0.379844,61,0.000000
1,raw_current_cnn,B,0.243659,-0.587296,paired_contrast,0.167344,0.562151,0.044573,0.006250,34,0.017009
2,morph_plus_raw_residual,C,0.831821,0.000866,paired_contrast,0.717935,0.947896,0.576406,0.396250,61,0.000960
3,shuffled_waveform_control,D,0.830519,-0.000436,paired_contrast,0.715996,0.948160,0.557840,0.380625,61,0.000889
4,corrected_q4n_diagnostic,E,0.833520,0.002565,paired_contrast,0.718274,0.950284,0.542920,0.373125,9,0.003551
5,comb_baseline_diagnostic,cleanComb,0.829777,-0.001178,mean_difference,0.713593,0.950746,0.531309,0.376250,9,0.000000


**해석.** morphology baseline(A) 의 k-sweep 달성률은 **0.8310**, 주 비교군 C 는 **0.8318** 다. `Δ vs A` 열은 같은 record·같은 seed 로 짝지은 대비이며, 짝지은 대비가 없는 arm 만 평균의 차이(\*)로 표기했다. seed SD 열은 seed 간 변동으로, 이 값보다 작은 차이는 해석하지 않는다.

In [31]:
display(Markdown("### 2. 주 비교 (확대)"))
show("primary_contrasts_zoom.png",
     f"**해석.** C−A = **{ca['mean']:+.4f}** [{ca['ci_low']:+.4f}, {ca['ci_high']:+.4f}], "
     f"C−D = **{cd['mean']:+.4f}** [{cd['ci_low']:+.4f}, {cd['ci_high']:+.4f}] 이다. "
     f"통과 기준은 평균 ≥ +{Q.GATE_MIN_GAIN} 이고 CI 하한 > 0 이므로, 점과 오차막대가 "
     f"파란 점선에 닿지 못하거나 CI 가 0(굵은 세로선)을 포함하면 실패다. "
     f"C−D 는 음성 대조로, 이 값이 0 이면 C 가 얻은 것은 박동 단위 파형 정보가 아니다.")

### 2. 주 비교 (확대)

**해석.** C−A = **+0.0009** [-0.0021, +0.0042], C−D = **+0.0013** [-0.0016, +0.0046] 이다. 통과 기준은 평균 ≥ +0.015 이고 CI 하한 > 0 이므로, 점과 오차막대가 파란 점선에 닿지 못하거나 CI 가 0(굵은 세로선)을 포함하면 실패다. C−D 는 음성 대조로, 이 값이 0 이면 C 가 얻은 것은 박동 단위 파형 정보가 아니다.

In [32]:
display(Markdown("### 3. 참조용 큰 차이 (축 분리)"))
show("reference_gap_separate.png",
     "**해석.** B−A(원파형 CNN 단독 − 형태 baseline)처럼 규모가 다른 대비는 "
     "여기 따로 그린다. 이전 `contrasts.png` 는 이 값과 주 비교를 같은 축에 그려서 "
     "±0.001 규모의 주 비교가 모두 0 근처의 한 점으로 뭉개졌다. "
     "축을 나눈 것은 표현의 문제일 뿐, 어떤 측정값도 바뀌지 않았다.")

### 3. 참조용 큰 차이 (축 분리)

**해석.** B−A(원파형 CNN 단독 − 형태 baseline)처럼 규모가 다른 대비는 여기 따로 그린다. 이전 `contrasts.png` 는 이 값과 주 비교를 같은 축에 그려서 ±0.001 규모의 주 비교가 모두 0 근처의 한 점으로 뭉개졌다. 축을 나눈 것은 표현의 문제일 뿐, 어떤 측정값도 바뀌지 않았다.

In [33]:
display(Markdown("### 4. k 별 달성률"))
show("achievement_by_k.png",
     f"**해석.** 왼쪽은 각 arm 의 achievement@k 이고, 오른쪽은 A 대비 C 의 차이를 "
     f"k 마다 확대한 것이다. 문헌 판독 운영점(k=30~50)에서의 거동이 임상적으로 가장 "
     f"중요하며, 형태 축의 이득은 k 가 작을수록 커지는 것이 지금까지의 패턴이었다. "
     f"오른쪽 패널은 seed 평균 간의 차이이므로 짝지은 신뢰구간이 아니다 — 판정은 "
     f"2번 그림의 CI 로만 한다.")

### 4. k 별 달성률

**해석.** 왼쪽은 각 arm 의 achievement@k 이고, 오른쪽은 A 대비 C 의 차이를 k 마다 확대한 것이다. 문헌 판독 운영점(k=30~50)에서의 거동이 임상적으로 가장 중요하며, 형태 축의 이득은 k 가 작을수록 커지는 것이 지금까지의 패턴이었다. 오른쪽 패널은 seed 평균 간의 차이이므로 짝지은 신뢰구간이 아니다 — 판정은 2번 그림의 CI 로만 한다.

In [34]:
display(Markdown("### 5. seed 별 방향"))
show("seed_effects.png",
     f"**해석.** {n_seed}개 seed 각각의 C−A 와 C−D 다. gate 4 는 "
     f"{Q.GATE_MIN_SEED_AGREE}/{n_seed} 이상의 seed 가 같은 양(+) 방향일 것을 요구하며, "
     f"이번 run 은 **{res['contrasts']['C_minus_A']['positive_seed_count']}/{n_seed}** 이다. "
     f"방향이 seed 마다 뒤집히면 평균이 양수여도 신뢰할 수 없다는 뜻이다.")

### 5. seed 별 방향

**해석.** 5개 seed 각각의 C−A 와 C−D 다. gate 4 는 4/5 이상의 seed 가 같은 양(+) 방향일 것을 요구하며, 이번 run 은 **4/5** 이다. 방향이 seed 마다 뒤집히면 평균이 양수여도 신뢰할 수 없다는 뜻이다.

In [35]:
display(Markdown("### 6. fold·seed 학습 진단"))
_warn = ("\n\n> ⚠️ **이 run 에서 Arm C 는 "
         f"{stall.get('n_best_epoch_zero')}/{stall.get('n_total')} (seed × fold) 전부에서 "
         "첫 번째 학습 epoch 완료 후의 체크포인트(`best_epoch = 0`)가 선택됐다.** "
         "이후 epoch 는 dev BCE 를 개선하지 못했다는 뜻이다. epoch 0 은 학습 전 상태가 "
         "아니다 - 한 epoch 분량(약 77~79 optimizer step)의 업데이트를 거쳤고, 선택된 "
         "체크포인트의 alpha 도 0 이 아니다(대체로 |0.078~0.101|). 이 run 은 학습 전 "
         "체크포인트(epoch -1)를 dev 후보로 평가하지 않았으므로, epoch 0 이 정확한 "
         "morphology baseline 보다 개선됐는지는 판정할 수 없다. alpha 부호는 head 부호와 "
         "함께 뒤집힐 수 있으므로 부호 자체를 seed 불안정성으로 읽지 말 것 - 해석 대상은 "
         "alpha × residual 출력이다.") if stall.get("all_zero") else ""
show("fold_training_diagnostics.png",
     "**해석.** 행은 seed, 열은 fold 다. `alpha` 는 학습된 잔차 스케일(0 에서 출발), "
     "`best_epoch` 는 early stopping 이 고른 epoch, `dev_loss` 는 그 지점의 dev BCE 손실이다. "
     "alpha 가 0 근처에 머물면 잔차가 사실상 꺼져 있다는 뜻이다." + _warn)

### 6. fold·seed 학습 진단

**해석.** 행은 seed, 열은 fold 다. `alpha` 는 학습된 잔차 스케일(0 에서 출발), `best_epoch` 는 early stopping 이 고른 epoch, `dev_loss` 는 그 지점의 dev BCE 손실이다. alpha 가 0 근처에 머물면 잔차가 사실상 꺼져 있다는 뜻이다.

> ⚠️ **이 run 에서 Arm C 는 25/25 (seed × fold) 전부에서 첫 번째 학습 epoch 완료 후의 체크포인트(`best_epoch = 0`)가 선택됐다.** 이후 epoch 는 dev BCE 를 개선하지 못했다는 뜻이다. epoch 0 은 학습 전 상태가 아니다 - 한 epoch 분량(약 77~79 optimizer step)의 업데이트를 거쳤고, 선택된 체크포인트의 alpha 도 0 이 아니다(대체로 |0.078~0.101|). 이 run 은 학습 전 체크포인트(epoch -1)를 dev 후보로 평가하지 않았으므로, epoch 0 이 정확한 morphology baseline 보다 개선됐는지는 판정할 수 없다. alpha 부호는 head 부호와 함께 뒤집힐 수 있으므로 부호 자체를 seed 불안정성으로 읽지 말 것 - 해석 대상은 alpha × residual 출력이다.

In [36]:
display(Markdown("### 7. 레코드(환자)별 delta"))
show("patient_delta_waterfall.png",
     f"**해석.** seed {n_seed}개를 **모두 평균한** record 별 C−A 를 정렬한 것이다"
     f"(한 seed 만 쓴 그림이 아니다). 파란색은 개선, 빨간색은 악화다. "
     f"평균이 0 이어도 개선과 악화가 서로 상쇄된 것인지, 아무 record 도 움직이지 "
     f"않은 것인지는 전혀 다른 이야기이며, 이 그림이 그것을 구분해 준다.")

pdf_ = pd.read_csv(os.path.join(FIG, "patient_delta.csv"))
display(Markdown("**개선 상위 10 record**"))
display(pdf_.nlargest(10, "delta_C_minus_A"))
display(Markdown("**악화 상위 10 record**"))
display(pdf_.nsmallest(10, "delta_C_minus_A"))
display(Markdown(
    f"**해석.** `s_burden` 은 그 record 의 S 비율, `ksw_A`/`ksw_C` 는 각 arm 의 성능이다. "
    f"개선·악화가 특정 burden 구간이나 baseline 성능 구간에 몰려 있는지 보라 — "
    f"몰려 있다면 다음 실험은 그 하위집단을 표적으로 삼아야 한다. "
    f"전체 {len(pdf_)}개 record 중 {int((pdf_['delta_C_minus_A'] > 0).sum())}개가 개선됐다."))

### 7. 레코드(환자)별 delta

**해석.** seed 5개를 **모두 평균한** record 별 C−A 를 정렬한 것이다(한 seed 만 쓴 그림이 아니다). 파란색은 개선, 빨간색은 악화다. 평균이 0 이어도 개선과 악화가 서로 상쇄된 것인지, 아무 record 도 움직이지 않은 것인지는 전혀 다른 이야기이며, 이 그림이 그것을 구분해 준다.

**개선 상위 10 record**

,record,fold,n_beat,n_s,s_burden,ksw_A,ksw_C,delta_C_minus_A,n_seed_averaged
55,62,0,2061,48,0.023290,0.536458,0.611458,0.075000,5
54,61,0,1910,64,0.033508,0.379844,0.396250,0.016406,5
53,53,2,2677,454,0.169593,0.896250,0.905583,0.009333,5
52,68,0,1959,399,0.203675,0.880000,0.887500,0.007500,5
51,51,4,3335,227,0.068066,0.499598,0.506699,0.007101,5
50,38,0,2537,227,0.089476,0.851244,0.857066,0.005822,5
49,21,1,1912,150,0.078452,0.926667,0.932333,0.005667,5
48,25,0,2558,104,0.040657,0.689712,0.695231,0.005519,5
47,30,2,1761,51,0.028961,0.875000,0.880000,0.005000,5
46,70,4,2688,125,0.046503,0.986000,0.990500,0.004500,5


**악화 상위 10 record**

,record,fold,n_beat,n_s,s_burden,ksw_A,ksw_C,delta_C_minus_A,n_seed_averaged
0,23,0,2384,44,0.018456,0.613636,0.573864,-0.039773,5
1,67,4,2881,53,0.018396,0.872547,0.860283,-0.012264,5
2,34,4,2627,32,0.012181,0.679688,0.670312,-0.009375,5
3,71,3,2311,68,0.029424,0.941618,0.933265,-0.008353,5
4,28,1,2884,27,0.009362,0.740741,0.733333,-0.007407,5
5,9,2,2543,112,0.044042,0.599107,0.592393,-0.006714,5
6,77,2,2346,80,0.034101,0.903125,0.896875,-0.006250,5
7,50,3,2999,136,0.045348,0.467647,0.462941,-0.004706,5
8,32,4,2139,74,0.034596,0.880811,0.876757,-0.004054,5
9,24,1,1851,106,0.057266,0.467406,0.464047,-0.003358,5


**해석.** `s_burden` 은 그 record 의 S 비율, `ksw_A`/`ksw_C` 는 각 arm 의 성능이다. 개선·악화가 특정 burden 구간이나 baseline 성능 구간에 몰려 있는지 보라 — 몰려 있다면 다음 실험은 그 하위집단을 표적으로 삼아야 한다. 전체 56개 record 중 24개가 개선됐다.

In [37]:
display(Markdown("### 8. 환자별 분포"))
show("metric_distribution.png",
     f"**해석.** A/C/D 의 record 별 k-sweep 분포다. 평균 하나로는 보이지 않는 "
     f"하위꼬리를 보기 위한 그림이며, 빨간 선이 p10(하위 10 백분위), 초록 선이 중앙값이다. "
     f"gate 5 는 C 의 p10 이 A 대비 {Q.GATE_LOWER_TAIL_MAX_DROP} 이상 나빠지지 않을 것을 "
     f"요구한다 — 평균이 올라도 최약 환자가 나빠지면 임상적으로 쓸 수 없기 때문이다.")

### 8. 환자별 분포

**해석.** A/C/D 의 record 별 k-sweep 분포다. 평균 하나로는 보이지 않는 하위꼬리를 보기 위한 그림이며, 빨간 선이 p10(하위 10 백분위), 초록 선이 중앙값이다. gate 5 는 C 의 p10 이 A 대비 0.01 이상 나빠지지 않을 것을 요구한다 — 평균이 올라도 최약 환자가 나빠지면 임상적으로 쓸 수 없기 때문이다.

In [38]:
display(Markdown("### 9. 학습 곡선"))
if report["training_history_present"]:
    show("learning_curves.png",
         "**해석.** (seed, fold) 마다 한 선이다. epoch 별 train/dev BCE 손실, dev PR-AUC, "
         "alpha 를 보여 준다. 체크포인트 선택은 **dev BCE 손실만** 사용하며, dev PR-AUC 는 "
         "기록 전용이라 선택에 개입하지 않는다.")
else:
    display(Markdown(
        "**이 run 에는 epoch 단위 training history 가 없다.** 기록 기능은 이번 개정에서 "
        "추가됐으므로 *이후* 실행부터 `training_history.json` 과 `learning_curves.png` 가 "
        "생성된다. 없는 데이터를 지어내지 않았고, 학습 곡선도 그리지 않았다."))

### 9. 학습 곡선

**이 run 에는 epoch 단위 training history 가 없다.** 기록 기능은 이번 개정에서 추가됐으므로 *이후* 실행부터 `training_history.json` 과 `learning_curves.png` 가 생성된다. 없는 데이터를 지어내지 않았고, 학습 곡선도 그리지 않았다.

## 9. report_summary.md

모든 핵심 수치, PASS/FAIL 근거, baseline 정의, Q4-N `0.8631` 을 제외한 이유, 구조 설명,
한계와 다음 결정, 생성한 그림 링크가 한 문서에 들어 있다.

In [39]:
display(Markdown(open(report["report_markdown"], encoding="utf-8").read()))

# EXP-2026-001 / Q4-O — 결과 보고서

- run: `20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn`
- 최종 판정: **NO-GO**
- 주 지표: record 단위 k-sweep 달성률 평균 (k = [50, 100, 200, 300])
- seed 5개 · record 56개 · fold 5개
- data sha256 `892f6ae9635db9bf715272c323a3c0e62e71693608bf66ca4dc9b66b69915a85`
- git commit `624e987b917ec021c9fc2130f37f6f35e720601c`

> 이 문서는 **표현(presentation) 전용**이다. 측정값은 `result.json` 에서
> 그대로 읽었고, arm·fold·seed·지표·bootstrap·gate 는 전혀 바꾸지 않았다.

## 1. Executive Summary

```text
==============================================================================
  EXP-2026-001 / Q4-O — Executive Summary
  run: 20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn
==============================================================================

■ 최종 판정: NO-GO
   사전 등록된 6개 gate 중 3개 통과 / 3개 실패.  (판정 기준은 실행 전에 고정되었고, 결과를 보고 바꾸지 않았다.)

■ morphology baseline (Arm A) — 이번 실험이 지켜낸 기준선
   k-sweep 달성률 평균  0.8310   (record 매크로 PR-AUC 0.7166 · AUROC 0.9483)
   하위꼬리 p10 0.5577 · 최악 레코드 61 (0.3798)
   ※ 이 값은 누수 없는 5-fold record-grouped CV 에서 측정됐다. Q4-N 의 0.8445/0.8631/0.8492 와 같은 자리에 두고 비교하면 안 된다.

■ 주 비교 — C(형태+원파형 잔차) − A(형태 단독)
   +0.0009 [95% CI -0.0021, +0.0042]
   seed 4/5 개에서 양(+) 방향 · 통과 기준은 평균 ≥ +0.015 이고 CI 하한 > 0

■ 음성 대조 — C − D(파형을 레코드 안에서 셔플한 동일 구조)
   +0.0013 [95% CI -0.0016, +0.0046]
   이 대비가 0 이면, C 가 얻은 것은 '박동 단위 파형 정보'가 아니다.

■ 진단 — E − cleanComb (Q4-N boost_fix 구조에서 잔차만의 효과)
   +0.0037 [95% CI -0.0035, +0.0118]

■ 통과한 gate
   ✅ 4_seed_direction_stable
   ✅ 5_lower_tail_not_worse
   ✅ 6_leakage_and_reproducibility
■ 실패한 gate
   ❌ 1_mean_gain_ge_0.015
   ❌ 2_ci_lower_gt_0
   ❌ 3_beats_shuffle_control

■ 이 결과가 의미하는 것
   · 누수를 제거한 조건에서, 현재 박동의 원파형 잔차는 형태 baseline 위에
     사전 등록한 크기(+0.015)의 이득을 주지 못했다.
   · C 가 파형 셔플 대조군 D 를 유의하게 이기지 못했다. 즉 C 의 점수는
     박동 단위 파형 정보에서 온 것이라고 말할 근거가 없다.
   · morphology baseline 은 유지된다. 이것이 현재까지 확립된 유일한 축이다.
   · Q4-N 의 boost_fix=0.8631 이 '개선'이 아니었다는 것과 정합적이다 —
     그 값은 80% 가 in-sample 인 offset 위에서 계산된 값이었다.

■ 이 결과가 증명하지 않는 것
   · '원파형에 S 판별 정보가 없다'는 것을 증명하지 않는다. 증명한 것은
     '이 구조·이 학습 스케줄·이 offset 에서 추가 이득이 없었다'는 것뿐이다.
   · ★ 중요 — Arm C 는 25개 (seed × fold) 전부에서
     첫 번째 학습 epoch 완료 후의 체크포인트(best_epoch = 0)가 선택됐고,
     이후 epoch 는 dev BCE 를 개선하지 못했다. epoch 0 은 학습 전 상태가
     아니다 — 한 epoch 분량(약 77~79 optimizer step)의 업데이트를 거친
     상태이며, 실제 선택된 체크포인트의 alpha 도 0 이 아니라 대체로
     |0.078~0.101| 이다. 이 run 은 학습 전 체크포인트(epoch -1)를 dev
     후보로 평가하지 않았으므로, epoch 0 이 정확한 morphology baseline
     보다 개선됐는지는 이 데이터만으로는 판정할 수 없다.
     학습률·epoch 수·checkpoint 선택 기준은 다음 실험의 1순위 점검 대상이다.
     (alpha 의 부호는 head 부호와 함께 뒤집힐 수 있으므로 부호 자체를
     seed 불안정성으로 읽지 말 것 — 해석 대상은 alpha × residual 출력이다.)
   · Transformer 나 더 큰 fusion 모델이 실패한다는 것을 증명하지 않는다.
     동시에, 그것을 시도할 근거가 생겼다는 뜻도 전혀 아니다.
   · MIT-BIH DS1→DS2 에서의 결과를 예측하지 않는다. 이 실험은 SVDB 다.

■ 권장 다음 행동
   1. Transformer·대형 fusion 모델로 가지 않는다 (사전 등록된 중단 규칙).
   2. morphology baseline 을 확정 기준선으로 고정하고 기록한다.
   3. 그 전에, 모든 (seed × fold) 가 첫 epoch 체크포인트에서 멈춘 것이
      스케줄 문제인지, 첫 epoch 이후 과적합인지, checkpoint 선택 기준
      문제인지 분리한다. 이는 새 가설이 아니라 이번 실행의 타당성
      점검이므로, 별도 spec 으로 사전 등록한다.
   4. 그다음 실패 레코드·하위꼬리 분석으로 돌아간다 (patient_delta_waterfall.png 참조).

■ 재현 정보 — data sha256 892f6ae9635db9bf… · git 624e987b91 · seeds [20260806, 20260807, 20260808, 20260809, 20260810]
==============================================================================
```

## 2. baseline 정의

**Arm A = morphology baseline** — Q4-N 의 `morph` arm 을 그대로 동결한 것(`F_BASE` RR 9열 ⊕ `MORPH` 형태 8열 = 17열), 로지스틱 회귀.
모든 scaler 와 model 은 outer-train 에서만 적합하고, outer-test 예측은
해당 test record 를 한 번도 보지 않은 모델이 만든다.

- k-sweep 달성률 평균 **0.8310**
- record 매크로 PR-AUC 0.7166 · AUROC 0.9483
- 하위꼬리 p10 0.5577 · 최악 레코드 #61 (0.3798)

이식 충실도 확인: 같은 특징을 Q4-N 의 LORO 프로토콜로 다시 채점하면 **0.8361** 로, Q4-N 이 보고한 `morph` 0.8361 와 delta -0.0000 이다 (허용치 0.005). 형태 특징 이식은 검증됐다.

## 3. Q4-N 의 0.8631 을 baseline 에서 제외한 이유

Q4-N 의 `cpu_fold()` 는 하나의 배열에 train 위치와 test 위치를 **둘 다** 썼고,
5개 fold 가 순차 실행되므로 뒤 fold 가 앞 fold 의 값을 덮어썼다.

```python
sc[tr] = lr.decision_function((X[tr] - mu) / sd)   # in-sample, 덮어써짐
sc[te] = lr.decision_function((X[te] - mu) / sd)
```

마지막 fold 이후 배열의 약 **80%** 가 in-sample 예측이다. 따라서
`cpu_comb=0.8445`, `boost_fix=0.8631`, `boost_rank=0.8492` 는 baseline 도
개선도 아니다. 잔차 CNN 은 학습 시 이미 그 박동들을 외운 offset 을 받았고,
테스트 시에는 깨끗한 offset 을 받았다 — offset 의 통계적 성격이 train 과
test 에서 서로 달랐다는 뜻이다. 이 값들은 Arm E 진단용 **오염된 참고값**
으로만 남긴다.

반대로 Q4-N 의 **CPU arm** (`morph − base = +0.1570` 등) 은 별도의 `loro()`
경로를 썼고 이 버그의 영향을 받지 않는다.

## 4. 구조 설명

- **A · morphology baseline (동결된 Q4-N 형태 특징, 로지스틱)**
- **B · 현재 박동 raw CNN (2리드 파형만)**
- **C · morphology + raw residual (주 비교군)**
- **D · 파형 셔플 대조군 (음성 대조)**
- **E · Q4-N boost_fix 구조 진단용 (누수 제거)**
- **cleanComb · comb 로지스틱 (Q4-N cpu_comb 의 깨끗한 대응물)**

`final_logit = morph_offset + alpha * cnn_residual` 이며 `alpha` 는 정확히 0
에서 출발한다(초기 *출력*이 baseline 과 동일하다는 뜻일 뿐, 학습 후 선택된
체크포인트의 성능 하한을 보장하지는 않는다). 잔차 head 는 xavier
초기화한다 — `alpha` 와 head 를 동시에 0 으로 두면 서로의 기울기를 0 에 가두는
Q4-N 의 초기화 데드락이 재현된다.

offset 은 각 outer fold 안에서 **inner cross-fitting** 으로 만든다:
outer-train 의 각 샘플은 자신을 학습에 쓰지 않은 inner 모델의 예측을 정확히
한 번 받고, outer-test offset 은 outer-train 전체로 적합한 모델이 만든다.

## 5. arm 요약

| arm | k-sweep | Δ vs A | PR-AUC | AUROC | p10 | worst (record) | seed SD |
|---|---|---|---|---|---|---|---|
| A | 0.8310 | — | 0.7166 | 0.9483 | 0.5577 | 0.380 (#61) | 0.0000 |
| B | 0.2437 | -0.5873 | 0.1673 | 0.5622 | 0.0446 | 0.006 (#34) | 0.0170 |
| C | 0.8318 | +0.0009 | 0.7179 | 0.9479 | 0.5764 | 0.396 (#61) | 0.0010 |
| D | 0.8305 | -0.0004 | 0.7160 | 0.9482 | 0.5578 | 0.381 (#61) | 0.0009 |
| E | 0.8335 | +0.0026 | 0.7183 | 0.9503 | 0.5429 | 0.373 (#9) | 0.0036 |
| cleanComb | 0.8298 | -0.0012 \* | 0.7136 | 0.9507 | 0.5313 | 0.376 (#9) | 0.0000 |

\* 짝지은 대비가 측정되지 않은 arm 은 평균의 차이로 표기했다.

## 6. PASS / FAIL 근거

| gate | 결과 | 측정값 |
|---|---|---|
| `1_mean_gain_ge_0.015` | ❌ FAIL | mean(C−A) = +0.0009 vs 기준 +0.015 |
| `2_ci_lower_gt_0` | ❌ FAIL | CI 하한 = -0.0021 |
| `3_beats_shuffle_control` | ❌ FAIL | mean(C−D) = +0.0013, CI 하한 -0.0016 |
| `4_seed_direction_stable` | ✅ PASS | 4/5 seed 양수, 기준 ≥ 4 |
| `5_lower_tail_not_worse` | ✅ PASS | p10 A 0.5577 → C 0.5764, 허용 하락 0.01 |
| `6_leakage_and_reproducibility` | ✅ PASS | 모든 누수 assertion 통과 (실패 시 실행 자체가 중단) |

**판정: NO-GO** — Keep the morphology baseline. Return to failure-record and lower-tail analysis. Do NOT build a Transformer or a larger fusion model.

## 7. 환자(레코드) 단위 결과

### 개선 상위 10

| record | S burden | n_S | A | C | Δ(C−A) |
|---|---|---|---|---|---|
| 62 | 0.0233 | 48 | 0.5365 | 0.6115 | +0.0750 |
| 61 | 0.0335 | 64 | 0.3798 | 0.3963 | +0.0164 |
| 53 | 0.1696 | 454 | 0.8962 | 0.9056 | +0.0093 |
| 68 | 0.2037 | 399 | 0.8800 | 0.8875 | +0.0075 |
| 51 | 0.0681 | 227 | 0.4996 | 0.5067 | +0.0071 |
| 38 | 0.0895 | 227 | 0.8512 | 0.8571 | +0.0058 |
| 21 | 0.0785 | 150 | 0.9267 | 0.9323 | +0.0057 |
| 25 | 0.0407 | 104 | 0.6897 | 0.6952 | +0.0055 |
| 30 | 0.0290 | 51 | 0.8750 | 0.8800 | +0.0050 |
| 70 | 0.0465 | 125 | 0.9860 | 0.9905 | +0.0045 |

### 악화 상위 10

| record | S burden | n_S | A | C | Δ(C−A) |
|---|---|---|---|---|---|
| 23 | 0.0185 | 44 | 0.6136 | 0.5739 | -0.0398 |
| 67 | 0.0184 | 53 | 0.8725 | 0.8603 | -0.0123 |
| 34 | 0.0122 | 32 | 0.6797 | 0.6703 | -0.0094 |
| 71 | 0.0294 | 68 | 0.9416 | 0.9333 | -0.0084 |
| 28 | 0.0094 | 27 | 0.7407 | 0.7333 | -0.0074 |
| 9 | 0.0440 | 112 | 0.5991 | 0.5924 | -0.0067 |
| 77 | 0.0341 | 80 | 0.9031 | 0.8969 | -0.0062 |
| 50 | 0.0453 | 136 | 0.4676 | 0.4629 | -0.0047 |
| 32 | 0.0346 | 74 | 0.8808 | 0.8768 | -0.0041 |
| 24 | 0.0573 | 106 | 0.4674 | 0.4640 | -0.0034 |

전체 표: [`figures/patient_delta.csv`](figures/patient_delta.csv) (모든 값은 seed 5개 평균)

## 8. 학습 진단

Arm C 의 early stopping 이 고른 epoch: 25/25 (seed × fold) 에서 `best_epoch = 0`.

> ⚠️ **전부 epoch 0 이다.** 모든 (seed × fold) 에서 **첫 번째 학습
> epoch 완료 후**의 체크포인트가 선택됐고, 이후 epoch 는 dev BCE 를
> 개선하지 못했다는 뜻이다. epoch 0 은 학습 전 상태가 아니다 — 한
> epoch 분량(약 77~79 optimizer step)의 업데이트를 거쳤고, 선택된
> 체크포인트의 alpha 도 0 이 아니다(대체로 |0.078~0.101|). 이 run 은
> 학습 전 체크포인트(epoch -1)를 dev 후보로 평가하지 않았으므로,
> epoch 0 이 정확한 morphology baseline 보다 개선됐는지는 판정할 수
> 없다. alpha 의 부호는 head 부호와 함께 뒤집힐 수 있으므로 부호를
> seed 불안정성으로 해석하지 말 것 — 해석 대상은 alpha × residual
> 출력이다.

**이 run 에는 epoch 단위 training history 가 없다.** history 기록 기능은
이번 개정에서 추가됐으므로 *이후* 실행부터 `training_history.json` 이
생성된다. 없는 데이터를 만들어내지 않았고, 학습 곡선도 그리지 않았다.

## 9. 한계와 다음 결정

- 이 결과는 **SVDB · record-grouped 5-fold** 한 프로토콜의 결과다. MIT-BIH DS1→DS2 를 예측하지 않는다.
- 절대값은 Q4-N 의 LORO 수치와 직접 비교할 수 없다(분할이 다르다). 해석 대상은 Q4-O 내부의 짝지은 대비뿐이다.
- NO-GO 는 Transformer 나 더 큰 fusion 모델을 시도할 근거가 아니다. 사전 등록된 중단 규칙이 이를 금지한다.
- 다음 결정의 1순위는 **best_epoch = 0 의 원인 분리**다(learning rate · 첫 epoch 이후 과적합 · checkpoint 선택 기준, 그리고 학습 전 상태(epoch -1)를 후보에 포함한 재평가). 이는 새 과학적 가설이 아니라 이번 실행의 타당성 확인이므로, 별도 spec 으로 사전 등록한 뒤 진행한다.
- 그다음은 실패 레코드·하위꼬리 분석이다.

## 10. 생성한 그림

- [`figures/arm_summary_table.png`](figures/arm_summary_table.png)
- [`figures/arm_metrics.csv`](figures/arm_metrics.csv)
- [`figures/primary_contrasts_zoom.png`](figures/primary_contrasts_zoom.png)
- [`figures/reference_gap_separate.png`](figures/reference_gap_separate.png)
- [`figures/achievement_by_k.png`](figures/achievement_by_k.png)
- [`figures/seed_effects.png`](figures/seed_effects.png)
- [`figures/fold_training_diagnostics.png`](figures/fold_training_diagnostics.png)
- [`figures/patient_delta_waterfall.png`](figures/patient_delta_waterfall.png)
- [`figures/patient_delta.csv`](figures/patient_delta.csv)
- [`figures/metric_distribution.png`](figures/metric_distribution.png)

## 11. 재현 확인

보고서가 저장된 logit 에서 다시 계산한 arm별·seed별 k-sweep 평균과 `result.json` 의 값의 최대 절대 오차: **0.000e+00** (30개 비교, 허용치 1e-09) → 일치


## 9. Bring the run back into GitHub

1. Save this executed notebook to `notebooks/quest47_q4o_leakage_free_residual_cnn.ipynb`
   and commit it — an unexecuted notebook is not evidence.
2. Ingest the measured result:

```bash
python pipelines/ingest_run.py \
    --results <run_dir>/result.json \
    --notebook notebooks/quest47_q4o_leakage_free_residual_cnn.ipynb
```

3. Register the run bundle in `research/ASSETS.md` (path, not a move).
4. Open the review PR with the exact commands and any deviations.

Do **not** update `research/PROJECT_STATE.md` with a new baseline until the design
owner has read the executed notebook and the measured result. Until then this
experiment has no outcome.

In [40]:
print("run bundle:", OUT_DIR)
for root, dirs, files in os.walk(OUT_DIR):
    for f_ in sorted(files):
        p = os.path.join(root, f_)
        print(f"  {os.path.relpath(p, OUT_DIR):<48} {os.path.getsize(p):>10,d} bytes")
Q.verify_bundle(OUT_DIR)
print("\nbundle schema verified")

run bundle: /content/drive/MyDrive/MedKOS/ecg-model/runs/20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn
  config.json                                           1,674 bytes
  fold_map.json                                         1,434 bytes
  log.txt                                              13,741 bytes
  manifest.json                                        24,829 bytes
  predictions.npz                                  12,868,261 bytes
  result.json                                          79,058 bytes
  figures/achievement_by_k.png                         91,683 bytes
  figures/arm_metrics.csv                               1,122 bytes
  figures/arm_summary_table.png                        86,768 bytes
  figures/arms_ksweep.png                              40,864 bytes
  figures/contrasts.png                                25,317 bytes
  figures/fold_training_diagnostics.png               250,997 bytes
  figures/metric_distribution.png                      79,756 bytes
  